# Retail Business Intelligence — SQL

A SQLite-based retail analytics project using CTEs, window functions, ranking, and profitability metrics.

> Dataset note: the database is synthetic and generated for portfolio practice.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_DIR = Path.cwd().parent
DB = PROJECT_DIR / 'data' / 'retail_analytics.db'
conn = sqlite3.connect(DB)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
tables

## Revenue by Category and Year-over-Year Growth

In [ ]:
query = '''
WITH yearly AS (
    SELECT p.category, strftime('%Y', o.order_date) AS year,
           ROUND(SUM(i.quantity * i.unit_price * (1 - i.discount)), 2) AS revenue
    FROM order_items i
    JOIN orders o ON i.order_id = o.order_id
    JOIN products p ON i.product_id = p.product_id
    WHERE o.status = 'Completed'
    GROUP BY p.category, year
)
SELECT category, year, revenue,
       ROUND((revenue - LAG(revenue) OVER (PARTITION BY category ORDER BY year)) * 100.0 /
             NULLIF(LAG(revenue) OVER (PARTITION BY category ORDER BY year), 0), 1) AS yoy_growth_pct
FROM yearly
ORDER BY category, year;
'''
pd.read_sql_query(query, conn)

## Customer Lifetime Value

In [ ]:
query = '''
SELECT c.customer_id, c.name, c.tier, c.city,
       COUNT(DISTINCT o.order_id) AS total_orders,
       ROUND(SUM(i.quantity * i.unit_price * (1 - i.discount)), 2) AS total_spend,
       RANK() OVER (ORDER BY SUM(i.quantity * i.unit_price * (1 - i.discount)) DESC) AS spend_rank
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items i ON o.order_id = i.order_id
WHERE o.status = 'Completed'
GROUP BY c.customer_id, c.name, c.tier, c.city
ORDER BY total_spend DESC
LIMIT 20;
'''
pd.read_sql_query(query, conn)

## Product Profitability

In [ ]:
query = '''
SELECT p.product_id, p.name, p.category,
       SUM(i.quantity) AS units_sold,
       ROUND(SUM(i.quantity * i.unit_price * (1 - i.discount)), 2) AS revenue,
       ROUND(SUM(i.quantity * p.unit_cost), 2) AS cogs,
       ROUND(SUM(i.quantity * (i.unit_price * (1 - i.discount) - p.unit_cost)), 2) AS gross_profit,
       ROUND(SUM(i.quantity * (i.unit_price * (1 - i.discount) - p.unit_cost)) * 100.0 /
             NULLIF(SUM(i.quantity * i.unit_price * (1 - i.discount)), 0), 1) AS margin_pct
FROM products p
JOIN order_items i ON p.product_id = i.product_id
JOIN orders o ON i.order_id = o.order_id
WHERE o.status = 'Completed'
GROUP BY p.product_id, p.name, p.category
ORDER BY gross_profit DESC
LIMIT 15;
'''
pd.read_sql_query(query, conn)

## Channel Performance

In [ ]:
query = '''
SELECT channel, COUNT(*) AS total_orders,
       SUM(status = 'Completed') AS completed,
       SUM(status = 'Returned') AS returned,
       SUM(status = 'Cancelled') AS cancelled,
       ROUND(AVG(status = 'Completed') * 100, 1) AS completion_rate
FROM orders
GROUP BY channel
ORDER BY total_orders DESC;
'''
pd.read_sql_query(query, conn)

## SQL Skills Demonstrated

CTEs, window functions (`LAG`, `RANK`), joins, conditional aggregation, date functions, grouping, profitability calculations, and KPI reporting.